In [ ]:
import pandas as pd

# Jaren waarvoor we ongevallen verwerken.
# Ongevallen kunnen voor alle jaren; het verkeer (INWEVA) van 2024 en 2025 staat in aparte bestanden (komt later).
JAREN = range(2018, 2026)

# Opbouw van dit notebook:
#   1. maak_baanvakken()            -> één keer: de snelwegbaanvakken uit INWEVA (netwerk van 2023)
#   2. verwerk_jaar(jaar, baanvakken) -> per jaar: ongevallen inlezen, filteren en koppelen aan een baanvak
#   3. alle jaren onder elkaar zetten in één tabel

In [ ]:
#0. data downloaden (alleen als die er nog niet is)
# download_data.py staat in dezelfde map als dit notebook. Het downloadt BRON 2018-2025 en INWEVA 2023-2025
# naar de map data/. Jaren die al aanwezig zijn worden overgeslagen; de eerste keer duurt dit even (enkele GB).
from download_data import download_alles

download_alles()

In [ ]:
#1. baanvakken maken (één keer)
# INWEVA gebruikt het wegennet van 2023. Daarom gebruiken we hier altijd wegvakken.txt van 2023,
# ook als we later ongevallen van andere jaren koppelen.

def maak_baanvakken():
    # INWEVA historie: één rij per baanvak (stuk rijbaan in één richting) met het verkeer per dag van 2018-2023
    jaarkolommen = [f"etmaal_AL_weekdag_{j}" for j in range(2018, 2024)]
    inweva = pd.read_excel("data/INWEVA/INWEVA_2023/A. Outputbestanden Excel/INWEVA_2023_historie.xlsx",
                           usecols=["VIAS_baannr", "Baantype_van", "Wegnr_van", "Ri_van", "Hm_van", "Hm_naar",
                                    "Lengte_baan_meter", "NwbId", "Traject_van", "Traject_naar"] + jaarkolommen)

    inweva_hr = inweva[inweva["Baantype_van"] == "HR"].copy()  # alleen hoofdrijbanen (geen op-/afritten)

    # wegennet 2023: welke wegvakken zijn een hoofdrijbaan van een snelweg?
    wegvakken = pd.read_csv("data/BRON/01-01-2023_31-12-2023/Netwerkgegevens/wegvakken.txt",
                            encoding="latin-1", dtype=str, usecols=["WVK_ID", "WEGBEHSRT", "BST_CODE", "ROUTELTR"])
    snelweg = wegvakken[
        (wegvakken["WEGBEHSRT"] == "R")      # Rijkswaterstaat
        & (wegvakken["BST_CODE"] == "HR")    # hoofdrijbaan
        & (wegvakken["ROUTELTR"] == "A")     # snelweg
    ]

    #alleen snelwegen filteren
    # INWEVA bevat ook N-wegen, maar heeft geen kolom "A of N".
    # Elk baanvak heeft wel een lijst NWB-wegvakken (NwbId). Staat een daarvan in 'snelweg', dan is het een snelweg.
    snelweg_ids = set(snelweg["WVK_ID"])   # set: Python kan hierin heel snel opzoeken

    def is_snelweg(nwb_ids):
        # "600461021H,508588042H" -> ["600461021", "508588042"]  (lijst maken, spaties en H/T eraf)
        ids = [x.strip().rstrip("HT") for x in nwb_ids.split(",")]
        return any(i in snelweg_ids for i in ids)   # True als minstens één wegvak een snelweg is

    baanvakken = inweva_hr[inweva_hr["NwbId"].apply(is_snelweg)].copy()

    #begin en eind van elk baanvak op volgorde zetten
    # Op de linkerbaan (L) lopen de kilometers af (Hm_van > Hm_naar). between() werkt alleen met (laag, hoog).
    baanvakken["hm_laag"] = baanvakken[["Hm_van", "Hm_naar"]].min(axis=1)
    baanvakken["hm_hoog"] = baanvakken[["Hm_van", "Hm_naar"]].max(axis=1)

    #baanvakken over een kilometersprong eruit
    # Bij een sprong (bijv. A10-ring begint weer bij km 0) is het bereik veel groter dan het baanvak echt lang is,
    # waardoor het ongevallen van elders zou "vangen".
    sprong = (baanvakken["hm_hoog"] - baanvakken["hm_laag"]) > baanvakken["Lengte_baan_meter"] + 1000
    baanvakken = baanvakken[~sprong].copy()   # ~ = niet: houd alles wat geen sprong is

    print(f"{len(baanvakken)} snelwegbaanvakken ({sprong.sum()} over een kilometersprong weggelaten)")
    return baanvakken

In [ ]:
baanvakken = maak_baanvakken()   # verwacht: 3209 snelwegbaanvakken (46 weggelaten)
baanvakken

In [ ]:
#2. één jaar ongevallen verwerken
# Alles gebruikt de bestanden van hetzelfde jaar (ongevallen, wegvakken, hectopunten),
# zodat de wegvak-ID's altijd bij elkaar passen.

def verwerk_jaar(jaar, baanvakken):
    map_bron = f"data/BRON/01-01-{jaar}_31-12-{jaar}"

    #data inlezen (alleen de kolommen die we nodig hebben)
    # ongevallen: nummer, wegvak, ernst (AP3_CODE), licht (LGD_ID), limiet (MAXSNELHD), type ongeval (AOL_ID)
    ongevallen = pd.read_csv(f"{map_bron}/Ongevallengegevens/ongevallen.txt", encoding="latin-1", dtype=str,
                             usecols=["VKL_NUMMER", "WVK_ID", "AP3_CODE", "LGD_ID", "MAXSNELHD", "AOL_ID"])
    # wegvakken: ID, beheerder, baansoort, routeletter (A/N), wegnummer, richting (R/L)
    wegvakken = pd.read_csv(f"{map_bron}/Netwerkgegevens/wegvakken.txt", encoding="latin-1", dtype=str,
                            usecols=["WVK_ID", "WEGBEHSRT", "BST_CODE", "ROUTELTR", "WEGNUMMER", "RPE_CODE"])

    #relevante rijen selecteren: alleen hoofdrijbanen van snelwegen
    snelweg = wegvakken[
        (wegvakken["WEGBEHSRT"] == "R")      # Rijkswaterstaat
        & (wegvakken["BST_CODE"] == "HR")    # hoofdrijbaan
        & (wegvakken["ROUTELTR"] == "A")     # snelweg
    ]
    snelweg = snelweg.drop(columns=["WEGBEHSRT", "BST_CODE", "ROUTELTR"])  # nu overal hetzelfde

    #ongevallen op de snelwegen
    # merge op WVK_ID plakt weg en richting aan elk ongeval; how="inner" houdt alleen snelwegongevallen
    ongevallen_snelweg = ongevallen.merge(snelweg, on="WVK_ID", how="inner")

    #twee kolommen in zelfde vorm als inweva toevoegen
    ongevallen_snelweg["weg"] = pd.to_numeric(ongevallen_snelweg["WEGNUMMER"])   # "004" -> 4

    #positie van het ongeval = midden van zijn wegvak
    # De kolom HECTOMETER is niet altijd ingevuld (2018: 36% leeg, 2023: 4% leeg), het wegvak (WVK_ID) wel.
    # Een INWEVA-baanvak bestaat uit hele wegvakken, dus het ongeval komt toch bij het juiste baanvak uit
    # (getest: 99,3-99,9% hetzelfde baanvak als met de echte hectometer). Zo werkt het voor elk jaar hetzelfde.
    hectopunten = pd.read_csv(f"{map_bron}/Netwerkgegevens/hectopunten.txt", encoding="latin-1", dtype=str,
                              usecols=["WVK_ID", "HECTOMETER"])
    hectopunten["meter"] = pd.to_numeric(hectopunten["HECTOMETER"]) * 100   # 156 -> 15600
    midden_wegvak = hectopunten.groupby("WVK_ID")["meter"].median()        # middelste paaltje per wegvak
    ongevallen_snelweg["positie_m"] = ongevallen_snelweg["WVK_ID"].map(midden_wegvak)  # opzoeken per ongeval

    #ongevallen koppelen aan baanvakken
    # Regel: een ongeval hoort bij het baanvak op dezelfde weg, in dezelfde richting,
    # waarvan het bereik (hm_laag tot hm_hoog) de positie van het ongeval bevat.
    # 1. merge op weg + richting: elk ongeval tegen ALLE baanvakken van zijn weg en richting
    kandidaten = ongevallen_snelweg.merge(
        baanvakken[["VIAS_baannr", "Wegnr_van", "Ri_van", "hm_laag", "hm_hoog"]],
        left_on=["weg", "RPE_CODE"], right_on=["Wegnr_van", "Ri_van"])
    # 2. alleen de rij houden waar de positie binnen het baanvak valt
    raak = kandidaten[kandidaten["positie_m"].between(kandidaten["hm_laag"], kandidaten["hm_hoog"])]
    # 3. ligt een ongeval precies op de grens van twee baanvakken, dan houden we er één
    gekoppeld = raak.drop_duplicates("VKL_NUMMER").copy()

    gekoppeld["jaar"] = jaar   # zodat we na het samenvoegen weten uit welk jaar elk ongeval komt

    print(f"{jaar}: {len(ongevallen_snelweg)} snelwegongevallen, {len(gekoppeld)} gekoppeld "
          f"({len(gekoppeld) / len(ongevallen_snelweg):.1%})")
    return gekoppeld

In [ ]:
#3. alle jaren verwerken en onder elkaar zetten
# [verwerk_jaar(j, baanvakken) for j in JAREN] maakt een lijst met één tabel per jaar;
# pd.concat plakt die onder elkaar tot één tabel. De kolom 'jaar' zegt uit welk jaar elk ongeval komt.
alle_ongevallen = pd.concat([verwerk_jaar(j, baanvakken) for j in JAREN], ignore_index=True)

# Resultaat: één rij per gekoppeld snelwegongeval, met het baanvak (VIAS_baannr) waar het gebeurde
alle_ongevallen

In [ ]:
#controle: in welk baanvak zit een ongeval?
# Ongeval 20230062249 -> baanvak 11553 (A2 rechts, km 222,15-224,19, tussen A73 en Echt)
voorbeeld = alle_ongevallen[alle_ongevallen["VKL_NUMMER"] == "20230062249"]
voorbeeld[["VKL_NUMMER", "jaar", "WEGNUMMER", "RPE_CODE", "positie_m", "VIAS_baannr", "hm_laag", "hm_hoog"]]

In [ ]:
#aantal ongevallen per baanvak per jaar
# groupby op baanvak én jaar, size() telt het aantal rijen (= ongevallen) per combinatie.
# unstack zet de jaren naast elkaar, zodat je per baanvak de jaren kunt vergelijken.
per_baanvak_jaar = alle_ongevallen.groupby(["VIAS_baannr", "jaar"]).size().unstack(fill_value=0)

# voorbeeld: baanvak 3901 (A4 rechts bij Hoofddorp) -> 2018: 27, 2019: 19, 2020: 6, ... 2023: 18
per_baanvak_jaar.loc[[3901]]